In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.datasets as datasets
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
from torch.optim.lr_scheduler import StepLR
from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter

from typing import Dict, Tuple
from tqdm import tqdm
import numpy as np
import time
import os
import random
from tabulate import tabulate

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib inline

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")



Torch version: 2.6.0
CUDA available: False
CUDA version: None
Number of GPUs: 0
GPU name: No GPU detected


In [10]:
from waveguide_dataset import WaveguideDataset
dataset = WaveguideDataset('train_test_split.h5')

In [11]:
class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

class Net4(nn.Module):
    """
    Deeper and wider CNN + FC architecture for regression on waveguide input with conditional features.
    """
    def __init__(self):
        super(Net4, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),    # [B, 64, 32, 32]
            nn.BatchNorm2d(64),
            nn.GELU(),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # [B, 128, 32, 32]
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.MaxPool2d(2),                                         # [B, 128, 16, 16]
            nn.Dropout(0.25),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1), # [B, 256, 16, 16]
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.MaxPool2d(2),                                         # [B, 256, 8, 8]
            nn.Dropout(0.25),

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1), # [B, 512, 8, 8]
            nn.BatchNorm2d(512),
            nn.GELU(),
            nn.MaxPool2d(2),                                         # [B, 512, 4, 4]
            nn.Dropout(0.25),

            Flatten()                                                # [B, 512 * 4 * 4 = 8192]
        )

        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048),
            nn.BatchNorm1d(2048),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.25),

            nn.Linear(256, 8)  # Output
        )

    def forward(self, x_img, x_cond):
        x = self.cnn(x_img)                 # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)   # [B, 8196]
        return self.fc(x)

In [12]:
class Net4_Mode0Weight0(nn.Module):
    """
    Net4 variant that predicts only:
    - mode 0  (original index 0)
    - weight0 (original index 4)
    Output shape: [B, 2]
    """
    def __init__(self):
        super().__init__()

        # ---------- CNN trunk (unchanged) ----------
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            Flatten()                               # -> [B, 8192]
        )

        # ---------- Fully-connected head ----------
        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048), nn.BatchNorm1d(2048), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(2048, 1024),     nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(1024, 256),      nn.BatchNorm1d(256),  nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, 2)                           # <- predict [mode0, weight0]
        )

    def forward(self, x_img, x_cond):
        x = self.cnn(x_img)                   # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)     # [B, 8196]
        return self.fc(x)                     # [B, 2]


In [13]:
def train(model, device, train_loader_in, optimizer, loss_fn):
    model.train()
    i = 0
    train_loader = tqdm(train_loader_in)
    for target, p, x in train_loader:
        i += 1
        target, p, x = target.to(device), p.to(device), x.to(device)
        optimizer.zero_grad()
        output = model(x, p)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        train_loader.set_description(f"loss: {loss.item():.4f}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Net4_Mode0Weight0().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1)


In [14]:
def test(model, device, test_loader, loss_fn, dataset, epoch_num, total_epoch):
    model.eval()
    test_loss = 0
    correct = 0
    samples = []

    with torch.no_grad():
        for target, p, x in tqdm(test_loader):
            target, p, x = target.to(device), p.to(device), x.to(device)
            output = model(x, p)
            test_loss += loss_fn(output, target).item() * x.size(0)
            for t, o, params in zip(target.cpu(), output.cpu(), p.cpu()):
                t_real = dataset.denormalize_cond(t)
                o_real = dataset.denormalize_cond(o)
                # params_real = dataset.denormalize_param(params)
                if len(samples) < 50:
                    samples.append((t_real.numpy(), o_real.numpy(), params.numpy()))#, params_real.numpy()))

    test_loss /= len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}\n')
    if epoch_num == total_epoch-1:
        chosen = random.sample(samples, 8)

        fig, axs = plt.subplots(2, 4, figsize=(16, 8))
        axs = axs.flatten()

        for i, (target, output, param) in enumerate(chosen):
            axs[i].plot(range(4), target[:4], 'r-o', label='Target')
            axs[i].plot(range(4), output[:4], 'b--o', label='Output')
            axs[i].set_title(f"Params: {param.round(2)}")
            axs[i].set_xlabel("Mode #")
            axs[i].set_ylabel("Value")
            axs[i].legend()
            axs[i].grid(True)

        plt.tight_layout()
        plt.show()
    return test_loss

In [15]:
def main(dataset):
    os.makedirs("models", exist_ok=True)
    batch_size = 128
    test_batch_size = 1000
    lr = 2e-3
    gamma = 0.9
    epochs = 200
    save_dir = 'only_top_mode'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = Net4_Mode0Weight0().to(device)
    # optimizer = optim.Adam(model.parameters(), lr=lr)
    # scheduler = StepLR(optimizer, step_size=1, gamma=gamma)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.MSELoss()

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=4)
    # pbar_train = tqdm(train_loader)
    # pbar_test = tqdm(test_loader)
    e_loss_graph = []
    for epoch in range(epochs):
        print(f'Epoch #{epoch}:')
        train(model, device, train_loader, optimizer, loss_fn)
        e_loss = test(model, device, test_loader, loss_fn, dataset, epoch, epochs)
        e_loss_graph.append(e_loss)
        scheduler.step()
        torch.save(model.state_dict(), f'models/{save_dir}.pth')
    plt.title(f"Epoch loss for {save_dir}")
    plt.plot(range(epochs), e_loss_graph)
    plt.xlabel('Epoch #')
    plt.ylabel("Loss on Test set")
if __name__ == '__main__':
    main(dataset)

Epoch #0:


  0%|          | 0/5601 [00:03<?, ?it/s]


KeyboardInterrupt: 